# EditGuard: one published VLKEB/MEND edit, save/reload, and rollback

This notebook tests the **feasibility of a published visual knowledge editor**. It does not train a new editor, benchmark a final test set, or guarantee a successful correction.

We use the authors' released BLIP-2/MEND checkpoint and one fixed **upstream TRAIN** example. That example may have been seen during MEND meta-training. Its answer is a counterfactual benchmark target, not a newly verified real-world fact. Four diagnostic questions are repeated across model states; they are not four independent editing experiments.

Select **Runtime → Change runtime type → GPU**, then run the cells in order. A T4 was available in our initial setup; another session can supply different hardware. Whether the complete model/edit fits is established by the smoke run, not by this notebook's existence. No Google Drive access is requested.

The run stops after one edit, four questions per state, same-process rollback, and fresh-process patch reload/rollback. Each model worker has a **1,800-second cap**. A timeout is not a runtime estimate. There is no meta-training, example search, or final evaluation here.

**Attribution and use:** [VLKEB](https://github.com/VLKEB/VLKEB), [MEND](https://arxiv.org/abs/2110.11309), and their released models supply the research method. Our code adds the bounded reproduction, integrity checks, generated-answer diagnostics, and patch lifecycle. OPT's model license is for **noncommercial research**; code, editor, annotations, and images have separate terms recorded in the asset manifest.

**Results provenance:** published numbers may be cited as upstream results, with their source and scoring protocol. They are not our measured results and must not be copied into local experiment output. This project has not meta-trained MEND. Our claims must follow our saved run evidence, including failed attempts.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile

REPOSITORY_URL = "https://github.com/ShivamRaghav7/verity.git"
WRAPPER_COMMIT = "a25a37034e6ff7ca1b73648405958e26b0e93959"
UPSTREAM_COMMIT = "10951b7b3788928f578b73f07eac9e1eaa0316f3"
REPOSITORY = Path("/content/editguard-vlkeb")
ENVIRONMENT = REPOSITORY / "artifacts/vlkeb/environment"
TOOLS = REPOSITORY / "artifacts/vlkeb/tools"
UPSTREAM = REPOSITORY / "tmp/vlkeb-upstream"
ASSETS = REPOSITORY / "artifacts/vlkeb/assets"
PYTHON = ENVIRONMENT / "bin/python"
# Keep defaults for historical compatibility; record variants as separate runs.
QUESTION_FORMAT = "raw"       # raw | vqa
TARGET_ALIGNMENT = "release"  # release | context
if QUESTION_FORMAT not in {"raw", "vqa"} or TARGET_ALIGNMENT not in {"release", "context"}:
    raise ValueError("Choose one of the explicitly supported question/alignment variants.")
STAGES = {}

if platform.system() != "Linux" or platform.machine().lower() != "x86_64" or not Path("/content").is_dir():
    raise RuntimeError("This notebook targets a Linux x86_64 Colab runtime, separate from the Windows application.")

def require_stage(name):
    if STAGES.get(name) is not True:
        raise RuntimeError(f"Complete the successful {name!r} cell first. A failed cell does not authorize later stages.")

def run_checked(command, *, cwd=None, env=None, timeout=1800):
    # No shell and no detached/background process. Output streams into this cell.
    print("Running:", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env,
                          text=True, check=True, timeout=timeout)

print("Configured a pinned development reproduction. No packages or models have been downloaded yet.")

## 1. Fetch our exact wrapper revision

The checkout is detached at a recorded commit. An existing checkout must already match it and have a clean tree. This cell does not reset or erase existing work. If you want a separate attempt using another revision, change `REPOSITORY` to a fresh directory and record the new commit deliberately.

In [ ]:
STAGES["checkout"] = False
if REPOSITORY.exists():
    if not (REPOSITORY / ".git").is_dir():
        raise RuntimeError(f"Existing target is not this notebook's Git checkout: {REPOSITORY}")
else:
    run_checked(["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, REPOSITORY], timeout=600)
    run_checked(["git", "-C", REPOSITORY, "fetch", "--depth", "1", "origin", WRAPPER_COMMIT], timeout=600)
    run_checked(["git", "-C", REPOSITORY, "checkout", "--detach", WRAPPER_COMMIT], timeout=600)
revision = subprocess.check_output(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True).strip()
dirty = subprocess.check_output(["git", "-C", str(REPOSITORY), "status", "--porcelain"], text=True).strip()
if revision != WRAPPER_COMMIT or dirty:
    raise RuntimeError(f"Existing checkout must be clean and at {WRAPPER_COMMIT}; found {revision}. No reset was performed.")
STAGES["checkout"] = True
print(json.dumps({"wrapper_commit": revision, "repository": str(REPOSITORY)}, indent=2))

## 2. Record the hardware actually assigned

This uses `nvidia-smi` and operating-system information, without loading a model. Available VRAM, host RAM and disk are separate constraints. The downloaded assets total **13,218,483,144 bytes (13.218 GB / 12.311 GiB)**, before dependencies, caches and exported patches. This disk check is necessary, not a complete storage or fit estimate.

In [ ]:
require_stage("checkout")
STAGES["hardware"] = False
query = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader,nounits"
], text=True, timeout=30)
gpus = [{"name": row[0].strip(), "total_vram_mib": int(row[1]),
         "free_vram_mib": int(row[2]), "driver": row[3].strip()}
        for row in csv.reader(io.StringIO(query)) if row]
if not gpus:
    raise RuntimeError("No NVIDIA GPU is assigned. Select a GPU runtime before continuing.")
meminfo = {}
for line in Path("/proc/meminfo").read_text().splitlines():
    key, value = line.split(":", 1)
    if key in {"MemTotal", "MemAvailable"}:
        meminfo[key + "_bytes"] = int(value.split()[0]) * 1024
manifest = json.loads((REPOSITORY / "configs/vlkeb-assets.json").read_text())
asset_bytes = sum(item["size_bytes"] for item in manifest["assets"])
disk = shutil.disk_usage(REPOSITORY)
inventory = {"utc": datetime.now(timezone.utc).isoformat(), "wrapper_commit": WRAPPER_COMMIT,
             "gpus": gpus, "host_memory": meminfo, "disk_free_bytes": disk.free,
             "declared_asset_bytes": asset_bytes,
             "limits": "Inventory only; no full model, edit, VRAM-fit or runtime claim."}
inventory_path = REPOSITORY / "artifacts/vlkeb/runtime-inventory.json"
inventory_path.parent.mkdir(parents=True, exist_ok=True)
inventory_path.write_text(json.dumps(inventory, indent=2) + "\n")
print(json.dumps(inventory, indent=2))
if disk.free < asset_bytes:
    raise RuntimeError("Free disk cannot hold the declared assets; dependencies and patches also need space.")
STAGES["hardware"] = True

## 3. Install the isolated environment and fetch upstream code

This cell explicitly downloads dependencies, including CUDA packages, and a separate Python 3.11.9. It leaves Colab's notebook Python and the existing Windows project environment alone. It fetches VLKEB at its exact pinned commit. It does **not** download models or run training.

A previous T4 session installed the initial dependencies in 57.568 seconds; the missing OpenCV processor dependency was then added, and the repaired selective import succeeded. Those observations do not establish a future setup time or full-model feasibility.

In [ ]:
require_stage("hardware")
STAGES["setup"] = False
run_checked([
    sys.executable, "scripts/bootstrap_vlkeb_colab.py", "--fetch-upstream", "--install",
    "--upstream-dir", UPSTREAM, "--env-dir", ENVIRONMENT, "--tool-dir", TOOLS
], cwd=REPOSITORY, timeout=3600)
bootstrap = json.loads((ENVIRONMENT / "bootstrap-manifest.json").read_text())
if not bootstrap["imports"].get("cuda_available"):
    raise RuntimeError("The isolated environment cannot access CUDA. Stop before downloading models.")
STAGES["setup"] = True
print(json.dumps(bootstrap["imports"], indent=2))

## 4. Import the real upstream model, editor and processor before weights

This checks the selected BLIP-2, MEND and image-processor modules. A basic `import torch` alone would miss processor dependencies such as `cv2`. A failure stops the workflow before model downloads. The check itself does not construct a model or deserialize a checkpoint.

In [ ]:
require_stage("setup")
STAGES["imports"] = False
check_script = r"""
import json, sys
from pathlib import Path
from ke_repro.bridge import _upstream_modules
modules = _upstream_modules(Path(sys.argv[1]))
report = {"status": "upstream_imports_passed", "modules": [module.__name__ for module in modules]}
Path(sys.argv[2]).write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))
"""
import_record = REPOSITORY / "artifacts/vlkeb/upstream-import.json"
run_checked([PYTHON, "-c", check_script, UPSTREAM, import_record], cwd=REPOSITORY, timeout=180)
STAGES["imports"] = True

## 5. Review the pinned asset plan

The manifest contains 25 assets totaling **13.218 GB**, including OPT, BERT, the EVA/BLIP-2 components, the released MEND editor, notices, training annotations and the source image archive. The complete image archive is fetched to extract only the fixed training fixture's four required images. Test annotation rows are not fetched or evaluated.

The next cell is a network-free plan. The separate download cell explicitly starts the large download. Checkpoint deserialization later uses trusted research files only after the recorded upstream size/checksum checks. Do not substitute an arbitrary `.pt`/`.pth` file.

In [ ]:
require_stage("imports")
run_checked([PYTHON, "scripts/prepare_vlkeb_assets.py", "--root", ASSETS], cwd=REPOSITORY, timeout=60)

## 6. Download, verify, and prepare the training-only fixture

Run this cell when ready for the **13.218 GB** download. Files use pinned source revisions and published integrity values; the two GCS components publish MD5 rather than SHA-256, and the preparation records local SHA-256 values as well. This cell does not load a model.

Interrupted `.part` downloads can resume while this runtime's disk survives. Rerunning skips already verified complete files. A lost Colab runtime loses its temporary disk; this notebook does not authorize Drive access or promise persistent storage. A checksum mismatch stops the process rather than substituting a newer file.

In [ ]:
require_stage("imports")
STAGES["assets"] = False
run_checked([PYTHON, "scripts/prepare_vlkeb_assets.py", "--download", "--root", ASSETS],
            cwd=REPOSITORY, timeout=7200)
run_checked([PYTHON, "scripts/prepare_vlkeb_assets.py", "--verify", "--root", ASSETS],
            cwd=REPOSITORY, timeout=900)
STAGES["assets"] = True

## 7. Fix the finite run and a fresh evidence directory

**Question:** can this released checkpoint execute one edit, generate short answers, persist the patch and restore its six editable matrices exactly on this runtime?

Inputs are the pinned upstream code, checked assets and fixed upstream training fixture. Outputs include `run_plan.json`, `preflight.json`, `edit.json`, `reload.json`, `smoke.json`, worker logs, and a saved patch. No target answer is passed to the generated-answer query function.

The first worker generates baseline, edited and rolled-back answers. A second process reloads the base model, applies the saved patch, generates answers, rolls back and generates again: **four cases in each of five answer states (20 generation calls)**. Each answer is capped at 24 new tokens, and each worker at 1,800 seconds. Exact runtime and peak memory are profiling outputs, not promises. No meta-training runs.

The output directory must not exist. Rerun this planning cell to choose a new directory after a failed attempt; the example, configuration and labels remain fixed. Existing evidence is preserved. Partial model state is not resumed; checked downloads are reusable.

### Explicit input variants

The defaults are **`QUESTION_FORMAT = "raw"`** and **`TARGET_ALIGNMENT = "release"`**, preserving the original attempt for comparison.

- `raw` sends the fixture question unchanged. `vqa` explicitly formats every edit/query as `Question: <question> Answer:`. The same format is used for all four cases and the fresh reload process. This is a different model-input configuration, not a correction silently added to a question.
- `release` uses the released collator's standalone target-token labels. `context` instead takes the actual target suffix from jointly tokenizing the question, a space and the target, and rejects any ambiguous retokenized question prefix. This is an experimental adapter change; it does not reproduce the original collation exactly.

The observed target used six standalone tokens but four contextual tokens. The release loss therefore selected a mismatched suffix span. Fixing the alignment is technically motivated, but the editor was trained using the release convention, so better answer quality is not guaranteed. Record each chosen combination and use a fresh run directory. The notebook does not sweep combinations or select examples by success.

The next planned development attempt uses `QUESTION_FORMAT = "vqa"` and `TARGET_ALIGNMENT = "context"` on this same fixture, followed by the full save/reload and rollback workflow. Select those values explicitly to run that configuration. The notebook defaults remain raw/release for historical compatibility; this plan does not imply a successful result.


In [ ]:
require_stage("assets")
STAGES["plan"] = False
RUN_ID = "smoke-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIRECTORY = REPOSITORY / "artifacts/vlkeb/runs" / RUN_ID
if RUN_DIRECTORY.exists():
    raise RuntimeError("Choose a fresh output directory; do not overwrite an earlier run.")
COMMON_ARGUMENTS = ["--upstream", UPSTREAM, "--assets", ASSETS,
                    "--manifest", REPOSITORY / "configs/vlkeb-assets.json",
                    "--output", RUN_DIRECTORY, "--device", "cuda",
                    "--max-new-tokens", "24", "--timeout-seconds", "1800",
                    "--question-format", QUESTION_FORMAT, "--target-alignment", TARGET_ALIGNMENT]
run_checked([PYTHON, "-m", "ke_repro", "plan", *COMMON_ARGUMENTS], cwd=REPOSITORY, timeout=60)
STAGES["plan"] = True
print("Fresh evidence directory:", RUN_DIRECTORY)

## 8. Execute the one-edit feasibility run

This is the first cell that loads the large models and edits parameters. It runs in the foreground and streams controller output. Each worker's detailed output goes to `edit.log` or `reload.log` in the evidence directory. You can inspect these files in Colab's Files panel while the cell is active.

A failed/OOM/timed-out run raises an error and retains its evidence. Do not read a failed cell as a success. The export cell below can still save failure reports. Passing the **technical gate** means edit/persistence/rollback consistency; the target answer can still be wrong, and unrelated answers can still change. Those outcomes are reported separately.

The [first actual T4 attempt](https://github.com/ShivamRaghav7/verity/blob/codex/upstream-knowledge-editor/reports/vlkeb/t4-smoke-1/smoke-1/smoke.json) **did not pass** with raw questions and release target alignment. Its edit worker completed in 128.027 seconds; its three same-process checks passed, but the direct generated answer missed the target, and the fresh-process reload was killed (exit -9). That code means the process was killed; memory pressure is plausible, but is not proven by the exit code alone. Peak sampled RSS was 11,912,224,768 bytes; CUDA allocator peaks were 10,946,286,080 allocated / 11,616,124,928 reserved bytes. The baseline answer was empty and the edited answer repeated “any”. These are measurements of that failed attempt, not expected results or proof of a useful edit.

A subsequent loading adaptation moves the already-loaded EVA encoder to CUDA earlier to reduce host-memory overlap. It preserves weights/dtypes and still needs a new profile. Context alignment and the VQA scaffold are separate, explicit diagnostic options; their success has not been established by the earlier failure.

The first runtime's temporary disk was later lost. Its JSON/log evidence was exported locally, but its approximately 629 MB patch was not retained. The first patch cannot be reconstructed or reloaded from its recorded hash alone. A new run must create a new patch and preserve its own evidence.


In [ ]:
require_stage("plan")
STAGES["smoke_passed"] = False
if RUN_DIRECTORY.exists():
    raise RuntimeError("This output directory already exists. Rerun the planning cell for a fresh attempt.")
STAGES["smoke_attempted"] = True
summary = None
try:
    run_checked([PYTHON, "-m", "ke_repro", "smoke", *COMMON_ARGUMENTS],
                cwd=REPOSITORY, timeout=3900)
finally:
    summary_file = RUN_DIRECTORY / "smoke.json"
    if summary_file.exists():
        summary = json.loads(summary_file.read_text())
        print(json.dumps({key: summary.get(key) for key in
                          ["execution_status", "technical_gate_passed", "direct_edit_target_matched",
                           "neighbor_answer_preserved", "accuracy_estimate", "interpretation"]}, indent=2))
    print("Evidence directory:", RUN_DIRECTORY)
if not isinstance(summary, dict) or summary.get("technical_gate_passed") is not True:
    raise RuntimeError("The technical gate did not pass; inspect and export the recorded failure.")
STAGES["smoke_passed"] = True

## 9. Read what actually happened

- **Technical gate:** eight checks covering changed editable matrices, exact rollback, regenerated answers, fresh-process base identity, and patch reload consistency.
- **Direct target match:** conservative normalized whole-answer matching. Mentioning a target inside a rejected possibility or a longer explanation is insufficient.
- **Paraphrase / alternate image:** separate diagnostic generations, not independent test edits.
- **Neighbor preserved:** the neighbor answer stayed the same. This does not prove the answer was factually correct; its author-target match is recorded separately.
- **Resources:** measured worker duration, sampled process RSS, and CUDA allocator peaks. These are profiling observations, not p50/p95 application latency.
- **Accuracy estimate = null:** intentional. One training-seen counterfactual edit cannot estimate held-out editing quality.

This summary cell also works after a failed smoke attempt if a summary was written. A process killed before writing JSON may leave only its worker log.

Any `teacher_forced_diagnostics` condition on the full target answer. They can help locate an alignment/decoding failure, but **must never be counted as independently generated-answer accuracy**. Read the recorded `target_alignment` and `question_format` alongside every run; compare variants as separate configurations.


In [ ]:
require_stage("plan")
if not RUN_DIRECTORY.is_dir():
    raise RuntimeError("The smoke run has not created an evidence directory yet.")
for filename in ["smoke.json", "edit.json", "reload.json"]:
    path = RUN_DIRECTORY / filename
    if not path.exists():
        print(filename, "was not written; inspect worker logs.")
        continue
    report = json.loads(path.read_text())
    print("\n" + filename)
    if filename == "smoke.json":
        print(json.dumps(report, indent=2))
    else:
        print(json.dumps({key: report.get(key) for key in
                          ["execution_status", "error_type", "error", "checks", "resources"]}, indent=2))
        for state in ["baseline", "edited", "restored", "loaded"]:
            if state in report:
                print(state, json.dumps([{key: row.get(key) for key in
                                         ["id", "question", "answer", "expected", "strict_target_match"]}
                                        for row in report[state]], indent=2))

## 10. Optional: diagnose a patch that still exists

Skip this section for the normal one-edit workflow. `ke_repro.diagnose` loads an **existing patch file** with matching base assets; a report or hash alone is insufficient. It cannot recover the lost first-run patch. The cell below defaults to **disabled**.

When enabled, it runs no new edit or training: one fresh model compares raw and VQA formats on the same four fixed questions before/after loading the patch (16 generations), records eight teacher-forced scores and two tokenization reports, then rolls back exactly. The worker has a 1,800-second cap. Its output goes to a separate fresh directory as `run_plan.json`, `preflight.json`, `diagnostic.json`, and `diagnostic.log`. Teacher-forced scores are not generated-answer accuracy. There is no example search or final evaluation.


In [ ]:
RUN_EXISTING_PATCH_DIAGNOSIS = False
if RUN_EXISTING_PATCH_DIAGNOSIS:
    require_stage("assets")
    require_stage("plan")
    EXISTING_PATCH = RUN_DIRECTORY / "patch"
    if not EXISTING_PATCH.is_file():
        raise FileNotFoundError("The actual patch file must still exist; JSON/hash evidence cannot reconstruct it.")
    DIAGNOSTIC_DIRECTORY = REPOSITORY / "artifacts/vlkeb/runs" / ("diagnose-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    if DIAGNOSTIC_DIRECTORY.exists():
        raise RuntimeError("Use a fresh diagnostic directory to preserve earlier evidence.")
    run_checked([PYTHON, "-m", "ke_repro.diagnose", "--upstream", UPSTREAM,
                 "--assets", ASSETS, "--manifest", REPOSITORY / "configs/vlkeb-assets.json",
                 "--patch", EXISTING_PATCH, "--output", DIAGNOSTIC_DIRECTORY,
                 "--device", "cuda", "--timeout-seconds", "1800"],
                cwd=REPOSITORY, timeout=1950)
    print("Separate diagnostic evidence:", DIAGNOSTIC_DIRECTORY)
else:
    print("Optional existing-patch diagnosis skipped; no model loaded or patch changed.")


## 11. Export the evidence, including failures (optional)

This creates a zip containing the run's JSON, logs and saved patch, plus hardware/setup/import records, the requirements, asset manifest and integrity records. **Full model weights, the dataset archive and environment packages are excluded.** A patch may still be sizable and requires the same compatible base assets/code; it is not a standalone model.

`DOWNLOAD_RESULTS = False` creates the archive on the runtime without opening a browser download. Set it to `True` and run the cell to download with Colab's file download mechanism. No Drive authorization is used. Keep the archive before disconnecting a temporary runtime.
The export below covers the main smoke run. If you explicitly run the optional retained-patch diagnosis, save its separate `DIAGNOSTIC_DIRECTORY` as well.


In [ ]:
DOWNLOAD_RESULTS = False
require_stage("plan")
if not RUN_DIRECTORY.is_dir():
    raise RuntimeError("There is no run directory to export.")
EXPORT_DIRECTORY = Path("/content/editguard-vlkeb-exports")
EXPORT_DIRECTORY.mkdir(exist_ok=True)
ARCHIVE = EXPORT_DIRECTORY / (RUN_DIRECTORY.name + ".zip")
if ARCHIVE.exists():
    raise RuntimeError(f"Archive already exists: {ARCHIVE}. Preserve it or select a new archive name.")
entries = []
for path in sorted(RUN_DIRECTORY.rglob("*")):
    if path.is_symlink():
        raise RuntimeError(f"Refusing to follow an unexpected export symlink: {path}")
    if path.is_file():
        entries.append((path, "run/" + path.relative_to(RUN_DIRECTORY).as_posix()))
context_files = [
    (REPOSITORY / "requirements-vlkeb.txt", "context/requirements-vlkeb.txt"),
    (REPOSITORY / "configs/vlkeb-assets.json", "context/vlkeb-assets.json"),
    (ENVIRONMENT / "bootstrap-manifest.json", "context/bootstrap-manifest.json"),
    (ENVIRONMENT / "resolved-requirements.txt", "context/resolved-requirements.txt"),
    (REPOSITORY / "artifacts/vlkeb/runtime-inventory.json", "context/runtime-inventory.json"),
    (REPOSITORY / "artifacts/vlkeb/upstream-import.json", "context/upstream-import.json"),
    (ASSETS / "asset-records.json", "context/asset-records.json"),
]
entries.extend((path, name) for path, name in context_files if path.is_file())
record = {"utc": datetime.now(timezone.utc).isoformat(), "wrapper_commit": WRAPPER_COMMIT,
          "upstream_commit": UPSTREAM_COMMIT, "run_id": RUN_DIRECTORY.name,
          "scope": "One fixed TRAIN counterfactual edit; no final evaluation or meta-training.",
          "question_format": QUESTION_FORMAT, "target_alignment": TARGET_ALIGNMENT,
          "files": []}
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path, name in entries:
        checksum = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                checksum.update(chunk)
        record["files"].append({"path": name, "size_bytes": path.stat().st_size,
                                "sha256": checksum.hexdigest()})
        archive.write(path, arcname=name)
    archive.writestr("export-manifest.json", json.dumps(record, indent=2) + "\n")
print(json.dumps({"archive": str(ARCHIVE), "size_bytes": ARCHIVE.stat().st_size,
                  "file_count": len(entries), "full_model_weights_included": False}, indent=2))
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(ARCHIVE))